# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 10 · Observed state, separate from forecast time

**Decision:** Round 3 completed, but no feature contrast passed. Do not repeat its eight fits or expand its failed pair-feature arms.

This notebook reviews the uploaded evidence, verifies the preserved control, and constructs two explicitly defined input groups. It fits **no new model**. The original 1,024-play selection, first-fold rows, labels and fixed estimator stay unchanged. The new feature preparation only needs plays used by that first fold.

The research question is not “can we add more columns?” It is whether directly exposing the observed state—and then compact landing-frame geometry—changes prediction quality under the same model capacity and training data. These are representations of known inputs, not additional raw information. The 0.46340 research target has not been reached by this package.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round4')
OUT = Path('/home/sagemaker-user/nfl-feature-round4-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open this kit in the existing NFL space with its existing Python environment.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'

def run(stage):
    command = [str(PY), str(KIT / 'run_round.py'), stage, '--out', str(OUT)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(2)
        process.wait(timeout=10)
        raise
    if code:
        raise RuntimeError(f'{stage} stopped (exit {code}). Preserve checkpoints and export the report. Do not change settings or reinstall packages.')

def show(fig, name):
    visuals.save(fig, OUT, name).show()


## 1 · Read the completed experiment

These are actual aggregates from your uploaded Round 3 report. They are not synthetic accuracy results. The historical precision error was superseded by successful preparation, fitting and exact no-refit replay. Intervals are exploratory on reused games.

In [ ]:
previous = visuals.read(KIT / 'evidence' / 'round3_summary.json')
show(visuals.scores(previous, 'Completed Round 3 · Same 6,326 rows / 16 evaluation games'), 'round3_scores')
show(visuals.contrasts(previous, 'Completed Round 3 · None of the planned feature gates passed'), 'round3_contrasts')
show(visuals.error_concentration(previous), 'round3_error_concentration')


## 2 · Verify the parent; do not refit it

The earlier adapter multiplied all 62 observed-state fields by `query_time + observation_age`. This includes role indicators and measurement support. The state is available separately, so we can test supplying it directly rather than asking a finite tree to reconstruct it from products.

This is a source-supported hypothesis, not a diagnosed accuracy bug. The first-place write-up uses separate dynamic/static inputs, but its architecture, training and ensemble also matter; features alone have not been shown to explain the whole gap.

Run the preflight. It verifies hashes, chronology, the old source, the existing locked numerical environment, and exact forward replay of **two existing control models**. It does not refit them or reinstall the project. A cached-runtime failure is a stop, not permission to upgrade packages.

In [ ]:
run('preflight')
preflight = visuals.read(OUT / 'preflight.json')
assert preflight['status'] == 'round4_preflight_passed'
print(json.dumps(preflight, indent=2))


## 3 · Construct the first 32 training plays

The feature builder has no target-value argument. Every reused legacy feature row must reproduce after its original float32/float64 storage conversion. Missing velocity and degenerate goal axes have explicit support flags. No raw output CSV is opened.

Require `state_smoke_complete` before preparing the rest.

In [ ]:
run('smoke')
smoke = visuals.read(OUT / 'smoke.json')
assert smoke['status'] == 'state_smoke_complete'
print({k: smoke[k] for k in ('plays', 'new_checkpoints', 'reused_checkpoints', 'direct_state_columns', 'goal_columns')})


## 4 · Finish first-fold feature preparation

Reuse the successful smoke checkpoints. No new play selection or additional fold is introduced. A completed checkpoint is immutable; a mismatch or orphan artifact must be diagnosed instead of overwritten. The preparation limit is 360 seconds with 15-second heartbeats.

In [ ]:
run('prepare')
prepared = visuals.read(OUT / 'preparation.json')
assert prepared['status'] == 'state_features_ready'
print({k: prepared[k] for k in ('plays', 'new_checkpoints', 'reused_checkpoints', 'parity_rows_rechecked', 'raw_output_csvs_opened')})


## 5 · Inspect support before fitting

The next figures use **training players only**, one count per player/play. A feature being present or variable is not evidence that it improves RMSE. The state ranges retain the existing fixed physical scaling. No normalization is fitted using evaluation outcomes.

In [ ]:
show(visuals.support(prepared), 'state_support')
show(visuals.state_ranges(prepared), 'direct_state_ranges')


## Checkpoint

Save this notebook with Ctrl+S. Only after all stages above pass should you open `11_state_geometry_ablation.ipynb`. It runs the four new coordinate fits and a separate no-fit replay. Do not rerun Round 3, modify thresholds, or open further-fold experiments.

In [ ]:
run('report')
print(OUT / 'nfl_feature_round4_report.zip')
